### 1. Environment Setup

**Python Version:** 3.10+

This cell installs the core stack:
- `chromadb`: Vector database for embedding storage and retrieval.
- `groq`: Fast LLM inference API.
- `sentence-transformers`: Local embedding generation.
- `pandas`: Data manipulation.
- `python-dotenv`: Environment variable management.

In [8]:
!pip install -q --upgrade chromadb groq sentence-transformers pandas python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 26.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.


In [9]:
# 1. Data Preparation and Indexing
data = [
    {"id": "1", "text": "High-performance laptop with 16GB RAM and long battery life for professionals.", "metadata": {"category": "electronics", "price": 1200}},
    {"id": "2", "text": "Portable Bluetooth speaker with waterproof design and deep bass.", "metadata": {"category": "accessories", "price": 50}},
    {"id": "3", "text": "Mechanical gaming keyboard with RGB lighting and tactile switches.", "metadata": {"category": "electronics", "price": 100}},
    {"id": "4", "text": "Budget smartphone with decent camera and expandable storage.", "metadata": {"category": "electronics", "price": 300}},
    {"id": "5", "text": "Leather laptop sleeve for 13-inch devices, stylish protection.", "metadata": {"category": "accessories", "price": 40}}
]

# Create or Reset Collection
try:
    chroma_client.delete_collection(name="hybrid_search_lab")
except:
    pass

collection = chroma_client.create_collection(name="hybrid_search_lab", embedding_function=embedding_func)

# Add documents to Chroma
collection.add(
    documents=[item["text"] for item in data],
    metadatas=[item["metadata"] for item in data],
    ids=[item["id"] for item in data]
)
print("Collection 'hybrid_search_lab' is ready.")

Collection 'hybrid_search_lab' is ready.


In [12]:
# 2. Hybrid Retrieval & LLM Generation
def hybrid_retrieve(query, category_filter=None, n_results=3):
    # Metadata filtering logic
    where_clause = {"category": category_filter} if category_filter else None

    # Semantic Search via ChromaDB
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_clause
    )
    return results

def generate_answer(query, results):
    docs = results['documents'][0]
    context = "\n".join(docs)

    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer concisely based on the context."

    # Use a currently supported model like llama-3.3-70b-versatile
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
    )
    return response.choices[0].message.content

In [13]:
# 3. Testing and Evaluation
test_queries = [
    ("laptop with long battery life", None),
    ("waterproof speaker", "accessories"),
    ("budget device", "electronics"),
    ("something made of leather", None),
    ("gaming peripherals", "electronics")
]

for query, cat in test_queries:
    print(f"\n>>> Query: '{query}' | Filter: {cat}")
    search_results = hybrid_retrieve(query, category_filter=cat)

    if search_results['documents'] and search_results['documents'][0]:
        print(f"Top Match: {search_results['documents'][0][0]}")
        answer = generate_answer(query, search_results)
        print(f"Groq Response: {answer}")
    else:
        print("No results found for this filter combination.")


>>> Query: 'laptop with long battery life' | Filter: None
Top Match: High-performance laptop with 16GB RAM and long battery life for professionals.
Groq Response: High-performance laptop with 16GB RAM.

>>> Query: 'waterproof speaker' | Filter: accessories
Top Match: Portable Bluetooth speaker with waterproof design and deep bass.
Groq Response: Portable Bluetooth speaker with waterproof design and deep bass.

>>> Query: 'budget device' | Filter: electronics
Top Match: Budget smartphone with decent camera and expandable storage.
Groq Response: Budget smartphone.

>>> Query: 'something made of leather' | Filter: None
Top Match: Leather laptop sleeve for 13-inch devices, stylish protection.
Groq Response: Laptop sleeve.

>>> Query: 'gaming peripherals' | Filter: electronics
Top Match: Mechanical gaming keyboard with RGB lighting and tactile switches.
Groq Response: Mechanical gaming keyboard with RGB lighting and tactile switches.


### 2. Hybrid Search Architecture

Our system follows a **Reciprocal Rank Fusion (RRF)** inspired logic or a weighted scoring mechanism:
1.  **Document Structure**: Each record contains `text` (content), and `metadata` (category, product_name, price).
2.  **Semantic Path**: Uses `all-MiniLM-L6-v2` to map queries to a 384-dimensional vector space.
3.  **Keyword Path**: Uses a boolean or BM25-style match within ChromaDB's built-in filtering.
4.  **Metadata Filter**: Applied at the database level to narrow the search space (e.g., searching only 'Electronics').
5.  **RAG**: The top-k results are injected into a Groq prompt for a grounded answer.

In [7]:
import os
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq
from google.colab import userdata
# Explicitly import sentence_transformers to ensure it's loaded properly
import sentence_transformers

# --- Configuration ---
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized.")
except Exception as e:
    print(f"Warning: Could not access secret GROQ_API_KEY: {e}")
    # Fallback for manual input if secret fails
    GROQ_API_KEY = "PASTE_YOUR_API_KEY_HERE"
    client = Groq(api_key=GROQ_API_KEY)

# Initialize ChromaDB
chroma_client = chromadb.Client()
model_name = "all-MiniLM-L6-v2"
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

print(f"Architecture initialized with embedding model: {model_name}")

Groq client initialized.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Architecture initialized with embedding model: all-MiniLM-L6-v2


### 3. Data Preparation and Indexing
We create a synthetic product catalog to test our hybrid logic.

In [14]:
data = [
    {"id": "1", "text": "High-performance laptop with 16GB RAM and long battery life for professionals.", "metadata": {"category": "electronics", "price": 1200}},
    {"id": "2", "text": "Portable Bluetooth speaker with waterproof design and deep bass.", "metadata": {"category": "accessories", "price": 50}},
    {"id": "3", "text": "Mechanical gaming keyboard with RGB lighting and tactile switches.", "metadata": {"category": "electronics", "price": 100}},
    {"id": "4", "text": "Budget smartphone with decent camera and expandable storage.", "metadata": {"category": "electronics", "price": 300}},
    {"id": "5", "text": "Leather laptop sleeve for 13-inch devices, stylish protection.", "metadata": {"category": "accessories", "price": 40}}
]

# Reset and Create Collection
try:
    chroma_client.delete_collection(name="hybrid_search_lab")
except:
    pass

collection = chroma_client.create_collection(name="hybrid_search_lab", embedding_function=embedding_func)

# Add documents to Chroma
collection.add(
    documents=[item["text"] for item in data],
    metadatas=[item["metadata"] for item in data],
    ids=[item["id"] for item in data]
)
print("Collection populated with metadata-rich documents.")

Collection populated with metadata-rich documents.


### 4. Hybrid Retrieval Engine
This function implements the hybrid logic: semantic search + metadata filtering.

In [4]:
def hybrid_retrieve(query, category_filter=None, n_results=3):
    # Prepare metadata filter for Chroma
    where_clause = {"category": category_filter} if category_filter else None

    # Vector search with optional metadata filtering
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_clause
    )

    return results

def generate_answer(query, context_docs):
    context = "\n".join(context_docs[0])
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer based ONLY on the context provided."

    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama3-8b-8192",
    )
    return chat_completion.choices[0].message.content

### 5. Testing and Evaluation
Let's test the system with various query patterns.

In [16]:
queries = [
    ("laptop with long battery life", None),            # Pure Semantic
    ("waterproof", "accessories"),                      # Hybrid + Filter
    ("budget phone", "electronics"),                   # Semantic + Category Filter
    ("mechanical keyboard", None),                     # Keyword-Heavy
    ("leather sleeve", "electronics")                  # Filter testing
]

for q_text, cat in queries:
    print(f"\n--- Query: {q_text} (Category: {cat}) ---")
    results = hybrid_retrieve(q_text, category_filter=cat)

    # Check if we have documents in the nested list structure returned by Chroma
    if results and results.get('documents') and results['documents'][0]:
        print(f"Top Match: {results['documents'][0][0]}")
        # Fix: Pass the whole results dictionary to match the function signature in cell 20ffe898
        # or ensure the function called matches the data passed.
        answer = generate_answer(q_text, results)
        print(f"Groq AI Response: {answer}")
    else:
        print("No documents found for this filter combination.")


--- Query: laptop with long battery life (Category: None) ---
Top Match: High-performance laptop with 16GB RAM and long battery life for professionals.
Groq AI Response: High-performance laptop.

--- Query: waterproof (Category: accessories) ---
Top Match: Portable Bluetooth speaker with waterproof design and deep bass.
Groq AI Response: The portable Bluetooth speaker has a waterproof design.

--- Query: budget phone (Category: electronics) ---
Top Match: Budget smartphone with decent camera and expandable storage.
Groq AI Response: Decent camera and expandable storage.

--- Query: mechanical keyboard (Category: None) ---
Top Match: Mechanical gaming keyboard with RGB lighting and tactile switches.
Groq AI Response: Mechanical gaming keyboard with RGB lighting and tactile switches.

--- Query: leather sleeve (Category: electronics) ---
Top Match: Mechanical gaming keyboard with RGB lighting and tactile switches.
Groq AI Response: For the high-performance laptop, a leather sleeve would